In [1]:
import pickle

In [2]:
with open('/root/autodl-tmp/chuandian_eq/data/taxi/raw/train.pkl', 'rb') as f:
    data = pickle.load(f)

In [3]:
print(data['train'][0])
print(len(data['train']))
print(data['dim_process'])

[{'idx_event': 1, 'type_event': 8, 'time_since_start': 0.0, 'time_since_last_event': 0.0}, {'idx_event': 2, 'type_event': 3, 'time_since_start': 0.07888888888888888, 'time_since_last_event': 0.07888888888888888}, {'idx_event': 3, 'type_event': 8, 'time_since_start': 0.27666666666666667, 'time_since_last_event': 0.19777777777777777}, {'idx_event': 4, 'type_event': 3, 'time_since_start': 0.37972222222222224, 'time_since_last_event': 0.10305555555555557}, {'idx_event': 5, 'type_event': 8, 'time_since_start': 0.5475, 'time_since_last_event': 0.16777777777777775}, {'idx_event': 6, 'type_event': 3, 'time_since_start': 1.0013888888888889, 'time_since_last_event': 0.4538888888888889}, {'idx_event': 7, 'type_event': 8, 'time_since_start': 1.4066666666666667, 'time_since_last_event': 0.40527777777777785}, {'idx_event': 8, 'type_event': 3, 'time_since_start': 1.8002777777777779, 'time_since_last_event': 0.39361111111111113}, {'idx_event': 9, 'type_event': 8, 'time_since_start': 1.8716666666666666

In [4]:
from src.data.sequence import Sequence
import torch
def list_of_dicts_to_sequence(event_list):
    inter_times = [event['time_since_last_event'] for event in event_list]
    inter_times = torch.tensor(inter_times, dtype=torch.float32)
    t_start = 0.0
    t_nll_start = 0.0
    arrival_times = [event['time_since_start'] for event in event_list]
    type_event = [event['type_event'] for event in event_list]
    type_event = torch.tensor(type_event, dtype=torch.long)
    return Sequence(
        t_start=t_start,
        t_nll_start=t_nll_start,
        arrival_times=arrival_times,
        inter_times=inter_times,
        type_event=type_event
    )


In [5]:
sequence_list = [list_of_dicts_to_sequence(s) for s in data["train"]]

/root/autodl-tmp/chuandian_eq/src/data/sequence.py:183: UserWarning: Found 1 zero inter-event times in the sequence. This violates fundamental assumptions of TPP models and may lead to incorrect log-likelihood values.
  warnings.warn(


In [6]:
from src.data.batch import Batch

In [7]:
from src.data.tpp_dataset import TppDataset
ds = TppDataset(sequence_list)
loader = ds.get_dataloader(
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

In [8]:
for batch in loader:
    print(batch.keys())
    break

['inter_times', 'arrival_times', 't_start', 't_end', 't_nll_start', 'nll_mask', 'start_idx', 'end_idx', 'non_pad_mask', 'type_seq', 'type_event']


In [9]:
batch.type_event[6]

tensor([5, 3, 8, 3, 8, 3, 8, 3, 8, 3, 8, 3, 8, 0, 5, 3, 8, 3, 8, 3, 8, 3, 8, 3,
        8, 3, 8, 3, 8, 3, 8, 3, 8, 3, 8, 0, 0, 0])

In [10]:
batch.type_seq

tensor([[   8,    3,    8,  ...,    0, -100, -100],
        [   5,    3,    8,  ...,    1,    8,    0],
        [   8,    3,    8,  ...,    3, -100, -100],
        ...,
        [   8,    3,    8,  ...,    3, -100, -100],
        [   8,    3,    8,  ...,    3,    8,    3],
        [   5,    0,    5,  ...,    0,    5,    0]])

In [ ]:
batch.arrival_times

tensor([[ 0.0000,  0.0950,  0.1272,  ..., 10.3350, 10.3350, 10.3350],
        [ 0.0000,  0.1061,  0.1961,  ...,  5.6961,  6.1336,  6.3978],
        [ 0.0000,  0.6267,  0.6544,  ...,  6.9356,  6.9356,  6.9356],
        ...,
        [ 0.0000,  0.0244,  0.0456,  ...,  6.7042,  6.7042,  6.7042],
        [ 0.0000,  0.0167,  0.0694,  ...,  6.3172,  6.3381,  6.5606],
        [ 0.0000,  0.2569,  0.7122,  ...,  6.4239,  6.4492,  6.5811]])

In [16]:
batch.inter_times

tensor([[0.0000, 0.0950, 0.0322,  ..., 0.9219, 0.0000, 0.0000],
        [0.0000, 0.1061, 0.0900,  ..., 0.2369, 0.4375, 0.2642],
        [0.0000, 0.6267, 0.0278,  ..., 0.0931, 0.0000, 0.0000],
        ...,
        [0.0000, 0.0244, 0.0211,  ..., 0.0811, 0.0000, 0.0000],
        [0.0000, 0.0167, 0.0528,  ..., 0.1136, 0.0208, 0.2225],
        [0.0000, 0.2569, 0.4553,  ..., 0.2628, 0.0253, 0.1319]])

In [12]:
from config.config_loader import load_args_from_yaml 
args = load_args_from_yaml("config/THP.yaml")
base_dir = f"data/{args.dataset}"

In [13]:
from src.data.preparation import prepare_data_tpp
df, train_loader, val_loader, test_loader,dataset = prepare_data_tpp(
    args,
    base_dir
)

ModuleNotFoundError: No module named 'src.catalogs.cd'

In [ ]:
for batch in test_loader:
    print(f"arival_times: {batch.arrival_times}")
    print(f"inter_times: {batch.inter_times}")
    # print(torch.max(batch.inter_times))

arival_times: tensor([[16472.0664, 16473.2090, 16473.3672,  ..., 16667.4121, 16667.8770,
         16670.0000],
        [16481.7559, 16485.3086, 16487.5449,  ..., 16680.0000, 16680.0000,
         16680.0000],
        [16490.0957, 16490.7988, 16495.5273,  ..., 16690.0000, 16690.0000,
         16690.0000],
        ...,
        [16760.5859, 16762.4688, 16765.3926,  ..., 16960.0000, 16960.0000,
         16960.0000],
        [16770.5625, 16774.7871, 16775.3711,  ..., 16970.0000, 16970.0000,
         16970.0000],
        [16780.3848, 16780.9004, 16781.5371,  ..., 16980.0000, 16980.0000,
         16980.0000]])
inter_times: tensor([[2.0664, 1.1427, 0.1572,  ..., 1.9021, 0.4656, 2.1230],
        [1.7559, 3.5533, 2.2361,  ..., 0.0000, 0.0000, 0.0000],
        [0.0957, 0.7028, 4.7281,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.5859, 1.8821, 2.9237,  ..., 0.0000, 0.0000, 0.0000],
        [0.5625, 4.2253, 0.5843,  ..., 0.0000, 0.0000, 0.0000],
        [0.3848, 0.5157, 0.6359,  ..., 0.000